# Hardware-test preflight setup

Run this notebook **before** `pytest tests/hardware/pertzlab --scope <scope>` to:

1. Instantiate the selected microscope with its real Micro-Manager cfg.
2. Open napari-micromanager sharing the same MMCore so you can live-view,
   focus, and try channels interactively.
3. Pick imaging / stim / optocheck channels from the cfg's actual groups.
4. Verify DMD-to-camera focus with a checkerboard snap.
5. Measure camera baseline (DMD off) and bright (DMD all-on) so the
   pytest tests can auto-calibrate their brightness thresholds instead
   of hardcoding them.

Results land in `tests/hardware/pertzlab/.preflight.json`, which `conftest.py`
reads at session start. Re-run this notebook whenever you change the
sample, LED power, binning, or camera ROI.

In [ ]:
# --- Cell 1: pick the scope ---
SCOPE = "moench"  # moench | niesen | jungfrau
# Written next to this notebook. Matches where conftest.py reads
# it from (``tests/hardware/pertzlab/.preflight.json``).
PREFLIGHT_PATH = ".preflight.json"

import json
from pathlib import Path

import numpy as np

In [ ]:
# --- Cell 2: instantiate scope (notebook owns mic.mmc) ---
if SCOPE == "moench":
    from faro.microscope.pertzlab.moench import Moench
    mic = Moench(affine_calibration_matrix=np.eye(3))
elif SCOPE == "niesen":
    from faro.microscope.pertzlab.niesen import Niesen
    mic = Niesen(affine_calibration_matrix=np.eye(3), fast_init=True)
elif SCOPE == "jungfrau":
    from faro.core.dmd import DMD
    from faro.microscope.pertzlab.jungfrau import Jungfrau
    mic = Jungfrau()
else:
    raise ValueError(f"unknown SCOPE {SCOPE!r}")

print(f"{SCOPE}: {mic.mmc.getLoadedDevices()}")

In [ ]:
# --- Cell 3: launch napari + hand our mmc to napari-mm ---
# napari-mm's MainWindow doesn't accept an ``mmcore`` kwarg, so it
# builds a throwaway CMMCorePlus during super().__init__. We then
# explicitly rewire via ``set_core`` — no singleton trickery, the
# data flow is notebook -> viewer.
import napari
from napari_micromanager import MainWindow

viewer = napari.Viewer()
win = MainWindow(viewer)
win.set_core(mic.mmc)
viewer.window.add_dock_widget(win, name="Micro-Manager")
print("napari-mm is now wired to the notebook's mic.mmc.")
print("Focus + preview, then return to the notebook for channel pinning.")

## Pick channels

After focusing in napari-mm, fill in the channel names below. The lists
come straight from the cfg so you know what's available. Exposures /
LED power can still be tuned later â€” they aren't written to
`.preflight.json` until the final cell.

In [ ]:
# --- Cell 4: enumerate available config groups + presets ---
for g in mic.mmc.getAvailableConfigGroups():
    presets = list(mic.mmc.getAvailableConfigs(g))
    print(f"  {g}: {presets}")

In [ ]:
# --- Cell 5: pin the selections. Edit these to match your setup. ---
CHANNEL_GROUP = "TTL_ERK"
IMAGING_CHANNEL = "mScarlet3"
OPTOCHECK_CHANNEL = "mCitrine"
STIM_CHANNEL = "CyanStim"
IMAGING_EXPOSURE_MS = 40.0
OPTOCHECK_EXPOSURE_MS = 50.0
STIM_EXPOSURE_MS = 50.0
STIM_POWER = 1  # LED level for the stim channel (e.g. Cyan_Level)

# Sanity: every named channel must exist in the cfg
_presets = set(mic.mmc.getAvailableConfigs(CHANNEL_GROUP))
for name in (IMAGING_CHANNEL, OPTOCHECK_CHANNEL, STIM_CHANNEL):
    assert name in _presets, f"{name!r} not in group {CHANNEL_GROUP!r}: {_presets}"
print("channels OK")

## Live DMD focus aid

The arm cell below latches a checkerboard onto the DMD, primes the
stim channel, and flips ``Mosaic3 OverlapMode`` to ``Off`` so the
pattern stays on continuously while napari's Live view streams.

Flow:

1. Run the arm cell.
2. Click **Live** in napari-mm. You should see crisp checker
   squares; if they're soft, refocus until sharp.
3. Stop Live in napari-mm.
4. Run the disarm cell — it blanks the DMD and restores
   ``OverlapMode`` so the brightness-calibration and MDA-based
   cells below behave as they do during the real pytest run
   (which needs OverlapMode=On for per-event DMD firing).

In [ ]:
# --- Cell 6a: arm DMD checkerboard for live focus ---
# Disable OverlapMode so the DMD latches the pattern continuously
# instead of waiting for per-frame camera TTL. OverlapMode=On is
# correct during the real MDA (tests pass under it), but napari's
# Live view doesn't reliably produce the TTL pulses the DMD expects,
# so in that context ``displaySLMImage`` behaves as a one-shot flash
# and the DMD sits in an unpredictable idle state between camera
# frames. Flipping to OverlapMode=Off makes the pattern visible
# throughout Live.
_prev_overlap = mic.mmc.getProperty(mic.dmd.name, "OverlapMode")
mic.mmc.setProperty(mic.dmd.name, "OverlapMode", "Off")

mic.mmc.setConfig(CHANNEL_GROUP, STIM_CHANNEL)
mic.mmc.setExposure(STIM_EXPOSURE_MS)
# Long SLM exposure so the pattern stays on as napari Live streams.
mic.mmc.setSLMExposure(mic.dmd.name, 60000.0)
try:
    mic.mmc.setProperty("LED", "Cyan_Level", STIM_POWER)
except Exception as e:
    print(f"(skipping LED power set: {e})")
mic.dmd.checker_board(pixels=100)
print(f"DMD checker armed (OverlapMode was {_prev_overlap!r}, now Off).")
print("Press Live in napari-mm and refocus. Run the next cell to disarm.")

In [ ]:
# --- Cell 6b: disarm DMD + restore OverlapMode ---
mic.dmd.all_off()
try:
    mic.mmc.setProperty(mic.dmd.name, "OverlapMode", _prev_overlap)
    print(f"DMD blanked, OverlapMode restored to {_prev_overlap!r}.")
except NameError:
    print("DMD blanked. (OverlapMode state unknown — arm cell wasn't run.)")

## Confirmation snap

One-shot via the same MDA path the pytest tests use — confirms the
checker is still crisp after disarming + re-arming through the full
event machinery.

In [ ]:
# --- Cell 6: checker-board focus aid (frameReady + imageSnapped block) ---
from threading import Event
from useq import MDAEvent, PropertyTuple
from faro.core._useq_compat import SLMImage
import matplotlib.pyplot as plt


def snap_direct(mask, channel, exposure_ms, power=None, power_device=None, power_prop=None):
    """Snap one camera frame while the DMD projects ``mask``.

    ``MDAEngine.exec_single_event`` does ``snapImage()`` then
    ``getImage()``. ``snapImage`` emits ``imageSnapped``, which fires
    napari-micromanager's handler synchronously — and napari's
    handler calls its own ``getImage``, consuming the camera buffer
    before the engine's ``getImage`` can. Block the signal for the
    duration of the snap so napari skips this frame; napari's viewer
    just won't update (fine for a brightness calibration that only
    needs the numeric pixel values).
    """
    if mic.mmc.isSequenceRunning():
        mic.mmc.stopSequenceAcquisition()
    mic.mmc.clearCircularBuffer()

    props = []
    if power is not None:
        props.append(PropertyTuple(
            device_name=power_device, property_name=power_prop, property_value=power
        ))
    event = MDAEvent(
        index={"t": 0, "p": 0},
        channel={"config": channel, "group": CHANNEL_GROUP},
        exposure=exposure_ms,
        min_start_time=0,
        properties=props or None,
        slm_image=SLMImage(device=mic.dmd.name, exposure=exposure_ms, data=mask),
        metadata={"stim": True},
    )
    done, captured = Event(), {}

    def on_frame(img, _evt, *_):
        captured["img"] = np.asarray(img)
        done.set()

    runner = mic.mmc.mda
    runner.events.frameReady.connect(on_frame)
    try:
        with mic.mmc.events.imageSnapped.blocked():
            thread = mic.mmc.run_mda(iter([event]))
            if not done.wait(timeout=15):
                try:
                    mic.mmc.mda.cancel()
                except Exception:
                    pass
                thread.join(timeout=5)
                raise RuntimeError(
                    "frameReady didn't fire within 15 s — check pymmcore "
                    "log for a driver-level error."
                )
            thread.join(timeout=10)
    finally:
        runner.events.frameReady.disconnect(on_frame)

    return captured["img"]


h, w = mic.dmd.height, mic.dmd.width
checker = ((np.indices((h, w)) // 100).sum(axis=0) % 2).astype(np.uint8) * 255
img = snap_direct(
    checker, STIM_CHANNEL, STIM_EXPOSURE_MS,
    power=STIM_POWER, power_device="LED", power_prop="Cyan_Level",
)
plt.figure(figsize=(6, 6))
plt.imshow(img, cmap="gray")
plt.title(f"checker snap  mean={img.mean():.0f}  max={img.max()}")
plt.axis("off")

## Brightness calibration

Snap one all-off and one all-on DMD frame, measure p99, and derive the
thresholds the brightness-based stim tests use.

In [ ]:
# --- Cell 7: baseline vs bright ---
blank = np.zeros((h, w), dtype=np.uint8)
allon = np.full((h, w), 255, dtype=np.uint8)

dark_img = snap_direct(blank, STIM_CHANNEL, STIM_EXPOSURE_MS,
                       power=STIM_POWER, power_device="LED", power_prop="Cyan_Level")
bright_img = snap_direct(allon, STIM_CHANNEL, STIM_EXPOSURE_MS,
                         power=STIM_POWER, power_device="LED", power_prop="Cyan_Level")

baseline_p99 = float(np.percentile(dark_img, 99))
bright_p99 = float(np.percentile(bright_img, 99))
bright_min_p99 = (baseline_p99 + bright_p99) / 2
bright_vs_dark_ratio = bright_p99 / max(baseline_p99, 1.0) / 2

print(f"baseline p99 (DMD off): {baseline_p99:.0f}")
print(f"bright   p99 (DMD on):  {bright_p99:.0f}")
print(f"BRIGHT_MIN_P99       -> {bright_min_p99:.0f}")
print(f"BRIGHT_VS_DARK_RATIO -> {bright_vs_dark_ratio:.1f}")
assert bright_p99 > 3 * baseline_p99, (
    "DMD all-on barely brighter than DMD off — check focus, LED power, "
    "stim channel, or whether room lights are on."
)

In [ ]:
# --- Cell 8: persist preflight JSON ---
x, y = mic.mmc.getXYPosition()
try:
    z = mic.mmc.getPosition()
except Exception:
    z = None

preflight = {
    "scope": SCOPE,
    "channel_group": CHANNEL_GROUP,
    "imaging_channel": IMAGING_CHANNEL,
    "imaging_exposure": IMAGING_EXPOSURE_MS,
    "optocheck_channel": OPTOCHECK_CHANNEL,
    "optocheck_exposure": OPTOCHECK_EXPOSURE_MS,
    "stim_channel": STIM_CHANNEL,
    "stim_exposure": STIM_EXPOSURE_MS,
    "stim_power": STIM_POWER,
    "baseline_p99": baseline_p99,
    "bright_p99": bright_p99,
    "bright_min_p99": bright_min_p99,
    "bright_vs_dark_ratio": bright_vs_dark_ratio,
    "origin": {"x": x, "y": y, "z": z},
}
Path(PREFLIGHT_PATH).write_text(json.dumps(preflight, indent=2))
print(f"wrote {PREFLIGHT_PATH}")
print(json.dumps(preflight, indent=2))

## Release hardware

Call ``mic.shutdown()`` before closing the notebook. Without it the
Python process hangs at exit waiting on device handles (same class
of bug as ``docs/napari_mm_device_leak.md``). Running this cell is
mandatory — a stuck kernel holding the Mosaic3 will block the
subsequent ``pytest`` run until you reboot.

In [ ]:
# --- Cell 9: release hardware ---
mic.shutdown()
print("mic shut down — safe to close the kernel and run pytest now.")

Now run `pytest tests/hardware/pertzlab --scope <scope>` from a fresh terminal.
**Close napari-mm first** â€” see `docs/napari_mm_device_leak.md` for why.